<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/transcription/09_Viterbi_HMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================
# [Bass Separator] Ultimate Environment Setup (v4.0 - Best Practice)
# ==================================================================
import os
import sys
import subprocess
from google.colab import drive

print("🚀 Bass Separator 통합 환경 설정을 시작합니다...")

# -----------------------------------------------------------------
# 1. Google Drive 마운트
# -----------------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# -----------------------------------------------------------------
# 2. GitHub 최신화 및 경로 설정 (충돌 없는 강제 동기화)
# -----------------------------------------------------------------
PROJECT_NAME = "Bass-separator"
REPO_URL = "https://github.com/sjkim-audio/Bass-separator.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

try:
    if not os.path.exists(PROJECT_PATH):
        print(f"📦 레포지토리 클론 중... ({PROJECT_NAME})")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    else:
        print(f"🔄 레포지토리 최신화 중... (Git Fetch & Reset)")
        subprocess.run(["git", "fetch", "--all"], cwd=PROJECT_PATH, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PROJECT_PATH, check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"❌ Git 동기화 실패: {e}")

# 핵심: 작업 디렉토리를 프로젝트 루트로 완벽히 고정하여 requirements.txt를 찾게 함
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

# -----------------------------------------------------------------
# 3. 커스텀 모듈(src) 실행을 통한 의존성 설치 및 데이터셋 로드
# -----------------------------------------------------------------
try:
    from src.env_setup import init_colab_env
    from src.utils import load_data_from_drive

    # env_setup.py 내부의 함수를 호출하여 requirements.txt 기반 설치 실행
    # (이 단계에서 torchcrepe, pretty_midi 등이 정상 설치됩니다)
    init_colab_env()

    # 데이터셋 복사
    MY_DRIVE_DATA_PATH = "/content/drive/MyDrive/Bass_separator/dataset"
    load_data_from_drive(MY_DRIVE_DATA_PATH, force_update=False)

except ImportError as e:
    print(f"⚠️ 커스텀 모듈 임포트 에러: {e}")
    print("   (src/utils.py의 'import shutil' 오타가 수정되었는지 확인하세요!)")
except Exception as e:
    print(f"❌ 셋업 중단: {e}")

# -----------------------------------------------------------------
# 4. 글로벌 라이브러리 사전 적재
# -----------------------------------------------------------------
import librosa
import numpy as np
import pandas as pd
import torchcrepe
import pretty_midi

print("\n🎉 Ready to Rock! 모든 셋업과 모듈 로드가 완벽히 끝났습니다.")

🚀 Bass Separator 통합 환경 설정을 시작합니다...
Mounted at /content/drive
📦 레포지토리 클론 중... (Bass-separator)
🚀 환경 설정을 시작합니다...

🔧 [시스템] 필수 도구 확인 중...
✅ FFmpeg가 이미 설치되어 있습니다.

🐍 [파이썬] 라이브러리 확인 중...
📄 requirements.txt 파일을 발견했습니다. 의존성 패키지를 설치합니다...
📦 패키지 일괄 설치 진행 중...
✅ 패키지 일괄 설치 완료.

🏥 설치 무결성 점검 (Health Check)...
✅ 필수 라이브러리가 모두 정상적으로 준비되었습니다!

🎉 모든 환경 설정이 완료되었습니다!
🚀 데이터 복사 시작...
   📂 Source: /content/drive/MyDrive/Bass_separator/dataset
   📂 Dest  : ./dataset
🎉 데이터 준비 완료! (총 5개 파일 복사됨)

🎉 Ready to Rock! 모든 셋업과 모듈 로드가 완벽히 끝났습니다.


In [10]:
import numpy as np
import librosa
from typing import List, Dict, Tuple, Any

class ViterbiSmartFingering:
    def __init__(self,
                 weight_fret: float = 1.0,
                 weight_string: float = 2.0,
                 shift_threshold: int = 3,
                 shift_penalty: float = 10.0,
                 open_string_penalty: float = 2.5): # 추가됨
        """
        시간 제약 및 톤 일관성이 추가된 Advanced Viterbi 디코더
        """
        self.w_f = weight_fret
        self.w_s = weight_string
        self.shift_thresh = shift_threshold
        self.shift_penalty = shift_penalty
        self.open_penalty = open_string_penalty

    def _calculate_transition_cost(self, pos1: Tuple[int, int], pos2: Tuple[int, int], dt: float) -> float:
        s1, f1 = pos1
        s2, f2 = pos2

        # 시간 가중치 계산 (최소 0.05초 보장)
        safe_dt = max(dt, 0.05)
        time_multiplier = 1.0 / safe_dt

        # 1. 수직 이동 비용 (줄 넘나들기)
        cost_s = self.w_s * abs(s2 - s1)

        # 2. 하이 프렛 자체 페널티 (Fret Height Penalty)
        # 물리적으로 프렛 번호가 높을수록 폼을 유지하기 어려우므로 기본 비용 부과
        cost_height = 0.5 * f2

        # 3. 개방현 관련 및 수평 이동 비용 계산
        cost_open = 0.0
        cost_f = 0.0

        if f1 == 0 and f2 != 0:
            # [수정] 개방현에서 닫힌 현으로 이동:
            # 거리를 0으로 면제하지 않고 도착 프렛(f2)에 비례하는 거리를 가상으로 계산하여 하이 프렛 도약 차단
            cost_f = self.w_f * f2 * 0.5 * time_multiplier

        elif f1 != 0 and f2 == 0:
            # 닫힌 현에서 개방현으로 진입 (톤 변화 이질감 페널티)
            cost_open = self.open_penalty

        elif f1 != 0 and f2 != 0:
            # 일반적인 프렛 이동
            dist_f = abs(f2 - f1)
            cost_f = self.w_f * dist_f * time_multiplier

            # 손가락 커버 범위를 벗어나는 포지션 이동 시 기하급수적 페널티
            if dist_f > self.shift_thresh:
                cost_f += self.shift_penalty * ((dist_f - self.shift_thresh) ** 2) * time_multiplier

        # 4. 동음 유지 보너스 (플래핑 억제)
        cost_stay = -2.0 if pos1 == pos2 else 0.0

        return cost_s + cost_height + cost_open + cost_f + cost_stay

    def decode(self, events: List[Dict[str, Any]], get_candidates_fn) -> List[Dict[str, Any]]:
        if not events:
            return []

        # 1. State Space 구성
        state_sequence = []
        for event in events:
            hz = librosa.midi_to_hz(event['midi_note']) if event['midi_note'] else 0
            candidates = get_candidates_fn(hz)
            if not candidates:
                candidates = [(0, 0)]
            state_sequence.append(candidates)

        n_steps = len(state_sequence)

        # 2. DP 테이블 초기화
        dp = [np.zeros(len(states)) for states in state_sequence]
        backpointers = [np.zeros(len(states), dtype=int) for states in state_sequence]

        # 3. Forward Pass
        for t in range(1, n_steps):
            prev_states = state_sequence[t-1]
            curr_states = state_sequence[t]

            # 두 노트 사이의 시간 계산 (단위: 초)
            dt = events[t]['time'] - events[t-1]['time']

            for curr_idx, curr_state in enumerate(curr_states):
                min_cost = float('inf')
                best_prev_idx = -1

                for prev_idx, prev_state in enumerate(prev_states):
                    trans_cost = self._calculate_transition_cost(prev_state, curr_state, dt)
                    total_cost = dp[t-1][prev_idx] + trans_cost

                    if total_cost < min_cost:
                        min_cost = total_cost
                        best_prev_idx = prev_idx

                dp[t][curr_idx] = min_cost
                backpointers[t][curr_idx] = best_prev_idx

        # 4. Backward Pass
        best_last_idx = int(np.argmin(dp[-1]))
        best_path_indices = [best_last_idx]

        for t in range(n_steps - 1, 0, -1):
            best_idx = backpointers[t][best_path_indices[-1]]
            best_path_indices.append(best_idx)

        best_path_indices.reverse()

        # 5. 최적화 결과 병합
        optimized_events = []
        for t, event in enumerate(events):
            opt_string, opt_fret = state_sequence[t][best_path_indices[t]]
            opt_event = event.copy()
            opt_event['string_idx'] = opt_string
            opt_event['fret'] = opt_fret
            optimized_events.append(opt_event)

        return optimized_events

In [11]:
import os
import librosa
from src.bass_transcription import get_f0_crepe_robust
from src.tab_generator import BassTabGenerator

# 앞서 정의한 ViterbiSmartFingering 클래스가 메모리에 선언되어 있어야 합니다.
# (ViterbiSmartFingering 클래스 정의 코드 실행 후 아래 코드 실행)

# 1. 파일 경로 설정 및 오디오 로드
audio_path = '/content/drive/MyDrive/Bass_separator/dataset/performance_test_demo(bass).wav'

if not os.path.exists(audio_path):
    raise FileNotFoundError(f"❌ 파일을 찾을 수 없습니다: {audio_path}")

print(f"📂 오디오 로드 중: {os.path.basename(audio_path)}")
y, sr = librosa.load(audio_path, sr=16000)

# 2. 피치 트래킹 (CREPE Tiny 모델)
print("🚀 피치 트래킹 실행 중...")
f0_data = get_f0_crepe_robust(y, sr, hop_length=160, model_capacity='tiny', batch_size=512)

# 3. 타브 악보 생성기 초기화 및 원시 이벤트 파싱
print("📝 기본 노트 이벤트 파싱 중...")
tab_gen = BassTabGenerator(sr=16000, hop_length=160)
tab_gen.parse_f0_to_events(f0_data)  # 내부적으로 임시 Greedy 탐색이 일어남

# 4. Viterbi 디코더 초기화 및 스마트 운지법 적용
print("🧠 Viterbi 기반 스마트 운지법 최적화 중...")
viterbi_decoder = ViterbiSmartFingering(
    weight_fret=1.0,
    weight_string=2.0,
    shift_threshold=3,
    shift_penalty=5.0
)

# 기존에 수집된 이벤트와 튜닝 정보(후보군 추출 함수)를 Viterbi 디코더에 전달
optimized_events = viterbi_decoder.decode(tab_gen.events, tab_gen.get_fret_candidates)

# 5. 최적화된 이벤트로 데이터 덮어쓰기 및 결과 렌더링
tab_gen.events = optimized_events
print("✨ 최적화 완료! 타브 악보를 렌더링합니다.")
tab_gen.display_tab(chars_per_line=80)

# (선택) 디버깅용 상위 5개 노트 비교 출력
print("\n🔍 [최적화된 이벤트 데이터 Sample (Top 5)]")
for idx, event in enumerate(tab_gen.events[:5]):
    print(f" - Note {idx+1}: Time={event['time']:.2f}s, String={tab_gen.string_names[event['string_idx']]}, Fret={event['fret']}, MIDI={event['midi_note']}")

📂 오디오 로드 중: performance_test_demo(bass).wav
🚀 피치 트래킹 실행 중...
⚠️ 경고: GPU가 감지되지 않아 연산이 매우 느려질 수 있습니다.
📝 기본 노트 이벤트 파싱 중...
🧠 Viterbi 기반 스마트 운지법 최적화 중...
✨ 최적화 완료! 타브 악보를 렌더링합니다.

🎸 Generated Bass Tab (Standard Tuning G-D-A-E)

G |----------------------------------------------------------------------------|
D |--0---------0--0------------------------------------------------------------|
A |---------------------------------------------3----3------------0----0-----0-|
E |--------------------3--------3--3--------------------------------------4----|

G |---------------------------------------------------------------------------|
D |-----0---------0--0--------------------------------------------------------|
A |--0----------------------------------------------3----3-----3-------0----0-|
E |-----------------------3-------4--3--3-------------------------------------|

G |--------------------------------------------------------------------|
D |------------------0--0----0----------0-----------0--3-